In [ ]:
# RQ6: Robustness and Generalization
# How robust is the best-performing model under different CV settings and train-test splits?

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, ShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/marketing-and-product-performance-dataset/marketing_and_product_performance.csv')
for col in ['Subscription_Tier', 'Common_Keywords']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df = df.drop(columns=['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID'])
X = df.drop(columns=['Units_Sold']).values
y = df['Units_Sold'].values
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

In [ ]:
results = []
for cv_folds in [3, 5, 10]:
    scores = cross_val_score(LinearRegression(), X_s, y, cv=cv_folds, scoring='r2')
    results.append({'Setting': f'{cv_folds}-Fold CV', 'Mean_R2': round(scores.mean(),4), 'Std_R2': round(scores.std(),4)})

for split in [0.1, 0.2, 0.3]:
    ss = ShuffleSplit(n_splits=10, test_size=split, random_state=42)
    scores = cross_val_score(LinearRegression(), X_s, y, cv=ss, scoring='r2')
    results.append({'Setting': f'Split={split}', 'Mean_R2': round(scores.mean(),4), 'Std_R2': round(scores.std(),4)})

res_df = pd.DataFrame(results)
print(res_df)
res_df.to_csv('RQ6_robustness.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(res_df))
ax.bar(x, res_df['Mean_R2'], yerr=res_df['Std_R2'], capsize=8, color='steelblue', alpha=0.8, ecolor='red')
ax.set_xticks(x)
ax.set_xticklabels(res_df['Setting'], fontsize=10)
ax.set_ylabel('R² Score')
ax.set_title('RQ6: Robustness — Linear Regression under Different CV and Split Settings', fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
for i, row in res_df.iterrows():
    ax.text(i, row['Mean_R2']+row['Std_R2']+0.002, f'{row["Mean_R2"]:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('RQ6_robustness.pdf', dpi=150, bbox_inches='tight')
plt.show()